# Demand lab — SKU × location
**Interview desk** · scripts-first · time-ordered evaluation

Framing for retail / supply-chain DS (RELEX-shaped): forecast is an *input*; value shows up in availability, waste, and bias under constraints. Work in clear functions; crystallize anything good into a runnable `.py` before submit.


## 0 · Environment


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore", category=DeprecationWarning)
rng = np.random.default_rng(42)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

ROOT = Path.cwd()
print("ROOT", ROOT.resolve())
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1 · Metrics that matter
Prefer **WMAPE** + **bias** over RMSE-only storytelling. Always keep a **baseline**.


In [ ]:
def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true).sum()
    return float(np.abs(y_true - y_pred).sum() / denom) if denom else float("nan")


def bias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float((y_pred - y_true).mean())


def time_split(df: pd.DataFrame, date_col: str = "date", holdout_days: int = 28):
    cut = df[date_col].max() - pd.Timedelta(days=holdout_days)
    train = df[df[date_col] <= cut].copy()
    test = df[df[date_col] > cut].copy()
    return train, test, cut


print("helpers ready: wmape / bias / time_split")

## 2 · Synthetic SKU-location panel
Replace with the live task data when it arrives. Structure mirrors day × SKU × store.


In [ ]:
def make_panel(n_days: int = 180, n_sku: int = 4, n_store: int = 3) -> pd.DataFrame:
    dates = pd.date_range("2025-01-01", periods=n_days, freq="D")
    rows = []
    for sku in range(n_sku):
        for store in range(n_store):
            base = 40 + 8 * sku + 5 * store
            season = 10 * np.sin(np.arange(n_days) / 7 * 2 * np.pi)
            noise = rng.normal(0, 3, n_days)
            promo = (rng.random(n_days) < 0.08).astype(float)
            y = np.clip(base + season + 12 * promo + noise, 0, None)
            rows.append(
                pd.DataFrame(
                    {
                        "date": dates,
                        "sku": f"S{sku}",
                        "store": f"L{store}",
                        "promo": promo,
                        "y": y,
                    }
                )
            )
    out = pd.concat(rows, ignore_index=True)
    out["sku_location"] = out["sku"] + "|" + out["store"]
    out["dow"] = out["date"].dt.dayofweek
    return out


df = make_panel()
train, test, cut = time_split(df)
print(f"rows={len(df):,}  sku_locations={df.sku_location.nunique()}  cut={cut.date()}")
print(f"train={len(train):,}  test={len(test):,}")
df.head(3)

## 3 · Baseline vs model (time-ordered)
Seasonal naive (lag-7) vs HistGradientBoosting on lag / calendar / promo features.


In [ ]:
def add_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()
    g["lag_7"] = g["y"].shift(7)
    g["lag_1"] = g["y"].shift(1)
    g["roll_7"] = g["y"].shift(1).rolling(7, min_periods=1).mean()
    return g


feat = df.groupby("sku_location", group_keys=False).apply(add_features)
feat = feat.dropna(subset=["lag_7", "lag_1"])
tr, te, cut = time_split(feat)
X_cols = ["lag_7", "lag_1", "roll_7", "dow", "promo"]

naive = te["lag_7"].to_numpy()
y_te = te["y"].to_numpy()

model = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.08, max_iter=120, random_state=42)
model.fit(tr[X_cols], tr["y"])
pred = model.predict(te[X_cols])

print(f"cut={cut.date()}")
print(f"seasonal_naive  WMAPE={wmape(y_te, naive):.4f}  bias={bias(y_te, naive):+.4f}")
print(f"HistGBR         WMAPE={wmape(y_te, pred):.4f}  bias={bias(y_te, pred):+.4f}")

## 4 · Quick visual (optional)


In [ ]:
sample = te[te["sku_location"] == te["sku_location"].iloc[0]].sort_values("date")
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(sample["date"], sample["y"], label="actual", lw=1.6)
ax.plot(sample["date"], model.predict(sample[X_cols]), label="HistGBR", lw=1.4)
ax.set_title(f"Holdout — {sample['sku_location'].iloc[0]}")
ax.legend(frameon=False)
ax.set_ylabel("units")
fig.tight_layout()
plt.show()

## 5 · Live task scratch
Drop the assignment data below. Keep evaluation **time-ordered**. When stable, move helpers into `solution.py` and run from Git Bash:

```bash
python solution.py
```


In [ ]:
# LIVE TASK — replace this cell
# 1. load data
# 2. define features (shift/rolling carefully — no leakage)
# 3. time_split
# 4. baseline + model
# 5. print WMAPE + bias (+ one plot if useful)
pass

## Notes / assumptions
- Grain: day × SKU × location unless the brief says otherwise
- Promo / price / weather: treat as drivers; watch leakage
- Finish artifact: runnable `.py` preferred for senior bar (see `FINLAND-BAR.md`)
